# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze Table

In [0]:
df = spark.table("databricks_medallion_pipeline.bronze.erp_loc_a101")
df.limit(10).display()

# Silver Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df.withColumn(field.name, trim(col(field.name)))

## Customer ID Cleanup

In [0]:
df = df.withColumn("cid", F.regexp_replace(col("cid"), "-", ""))

##Country Normalization

In [0]:
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry"))
)

# Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Verifying Dataframe and Saving

## Printing rows for verification

In [0]:
df.limit(10).display()

##Writing Silver Table

In [0]:
df.write.mode("overwrite").saveAsTable("databricks_medallion_pipeline.silver.erp_customer_location")

#Verifying Saved Table

In [0]:
%sql
SELECT * FROM databricks_medallion_pipeline.silver.erp_customer_location